# Semana 01 · Ciclo de vida completo de un modelo con MLflow

En esta práctica llevarás un caso de clasificación desde los datos hasta una API local:

`datos → experimentos → comparación → gate → modelo ganador → Registry → API → observabilidad`

El objetivo no es conseguir el número más alto, sino construir una decisión **reproducible,
auditable y desplegable**. El dataset cardiovascular es únicamente didáctico: el resultado
no puede usarse para diagnóstico, tratamiento ni decisiones sobre pacientes.

**Entorno:** Databricks Free Edition, compute serverless y experimento asociado a este notebook.
No necesitas tarjeta de crédito, cluster propio, Docker, `uv`, `pyproject.toml` ni tokens.

## Mapa y criterios de terminación

Al acabar debes poder enseñar en MLflow:

1. un lote de al menos seis runs comparables, con datos, parámetros, métricas y artefactos;
2. una regla de selección escrita **antes** de mirar el test;
3. un único candidato evaluado sobre test y marcado como ganador;
4. versiones en Unity Catalog y un ciclo `Challenger → Champion → rollback`;
5. una API HTTP local que carga el `model.pkl` del run ganador y supera pruebas válidas e inválidas;
6. un run de despliegue con latencia, volumen de peticiones, errores y evidencia de la respuesta.

Tiempo recomendado: 45 min de construcción guiada + 120–180 min de práctica autónoma y debrief.

## 0. Dependencias

Ejecuta `%pip` al principio. Si Databricks reinicia Python, continúa desde la celda de imports.
Las dependencias se instalan en la sesión del notebook; no se crea ningún entorno o fichero de
proyecto en el repositorio.

In [0]:
%pip install -q --upgrade "mlflow[databricks]==3.14.0" "scikit-learn==1.9.0" "pandas==3.0.3" "matplotlib==3.11.1"

## Cómo trabajar con la versión sin resolver

Ejecuta las celdas en orden y completa cada `TODO`. Los bloques se detienen con
`NotImplementedError` para que un fallo no genere evidencia engañosa. Consulta la guía del
alumno antes de mirar la solución. Mantén el test cerrado hasta haber elegido el ganador.

## 1. Configuración e identidad del lote

In [0]:
import json
import pickle
import re
import tempfile
import threading
import time
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import sklearn
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, classification_report,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

STUDENT_ALIAS = "carrascoalvarezgonzalo@gmail.com"
BATCH_ID = uuid.uuid4().hex[:8]
RANDOM_STATE = 42
MIN_VALIDATION_RECALL = 0.72
DATASET_PATH = "../../../data/raw/heart.csv"


assert re.match(r"^[a-zA-Z0-9.\-@]{5,50}$", STUDENT_ALIAS)

# Configura el tracking de MLflow en Databricks
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(f"/Users/{STUDENT_ALIAS}/s01-lifecycle-{BATCH_ID}")
experiment_id = mlflow.get_experiment_by_name(f"/Users/{STUDENT_ALIAS}/s01-lifecycle-{BATCH_ID}").experiment_id

2026/09/30 10:47:20 INFO mlflow.tracking.fluent: Experiment with name '/Users/carrascoalvarezgonzalo@gmail.com/s01-lifecycle-acf33af9' does not exist. Creating a new experiment.


## 2. Contrato y controles de datos

Permite nulos porque el pipeline los imputará, pero detente si faltan columnas, el dataset está
vacío o el target no es binario. Construye un `quality_report` serializable como JSON.

In [0]:
data = pd.read_csv(DATASET_PATH)
TARGET = "target"
EXPECTED_FEATURES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal",
]

# TODO 2: valida el contrato y crea X, y y quality_report.
# Debe incluir filas, duplicados, distribución del target, nulos y checks ejecutados.
X = data[EXPECTED_FEATURES]
y = data[TARGET]
quality_report = {
    "filas": len(data),
    "duplicados": data.duplicated().sum(),
}
assert quality_report["filas"] == len(data), "filas"
assert quality_report["duplicados"] == data.duplicated().sum()


## 3. Divide 60 % train, 20 % validación y 20 % test, siempre estratificado

In [0]:
# No uses X_test ni y_test hasta la sección 9.
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE)


## 4. Pipeline y funciones de evaluación

In [0]:
categorical_columns = X.select_dtypes(include=["object", "bool"]).columns.tolist()
numeric_columns = [c for c in EXPECTED_FEATURES if c not in categorical_columns]


def build_pipeline(config: dict) -> Pipeline:
    numeric_transformer = SimpleImputer(strategy="median")
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_columns),
            ("cat", categorical_transformer, categorical_columns),
        ]
    )

    rf = RandomForestClassifier(
        n_estimators=config.get("n_estimators", 100),
        max_depth=config.get("max_depth", None),
        min_samples_leaf=config.get("min_samples_leaf", 1),
        class_weight=config.get("class_weight", None),
        random_state=config.get("random_state", RANDOM_STATE),
    )

    return Pipeline(steps=[("preprocessor", preprocessor), ("classifier", rf)])


def evaluate_classifier(model: Pipeline, features: pd.DataFrame, labels: pd.Series) -> dict:
    start = time.perf_counter()
    predictions = model.predict(features)
    elapsed = time.perf_counter() - start
    latency_per_row = elapsed / len(features) if len(features) > 0 else 0.0

    probabilities = None
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(features)[:, 1]

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
        "roc_auc": roc_auc_score(labels, probabilities) if probabilities is not None else None,
        "latency_per_row": latency_per_row,
    }

## 5. Diseña al menos seis candidatos comparables

In [0]:

CANDIDATES = [
    {"name": "rf_base",          "n_estimators": 100, "max_depth": None, "min_samples_leaf": 1, "class_weight": None,        "random_state": RANDOM_STATE},
    {"name": "rf_deep",          "n_estimators": 100, "max_depth": 20,  "min_samples_leaf": 1, "class_weight": None,        "random_state": RANDOM_STATE},
    {"name": "rf_shallow",       "n_estimators": 100, "max_depth": 5,   "min_samples_leaf": 1, "class_weight": None,        "random_state": RANDOM_STATE},
    {"name": "rf_more_trees",    "n_estimators": 300, "max_depth": None, "min_samples_leaf": 1, "class_weight": None,        "random_state": RANDOM_STATE},
    {"name": "rf_leaf5",         "n_estimators": 100, "max_depth": None, "min_samples_leaf": 5, "class_weight": None,        "random_state": RANDOM_STATE},
    {"name": "rf_balanced",      "n_estimators": 100, "max_depth": None, "min_samples_leaf": 1, "class_weight": "balanced", "random_state": RANDOM_STATE},
    {"name": "rf_deep_balanced", "n_estimators": 200, "max_depth": 15,  "min_samples_leaf": 3, "class_weight": "balanced", "random_state": RANDOM_STATE},
    {"name": "rf_pruned",        "n_estimators": 150, "max_depth": 10,  "min_samples_leaf": 4, "class_weight": None,        "random_state": RANDOM_STATE},
]
assert len(CANDIDATES) >= 6, "Necesitas al menos seis candidatos."

## 6. Tracking profundo: un run por candidato

Por cada candidato registra tags, parámetros, versiones, inputs de train/validación, tarjeta de
datos, calidad, riesgos, métricas, reporte por clase, matriz de confusión, modelo MLflow con
firma/input example, `deployment/model.pkl` y contrato de inferencia.

In [0]:
dataset_card = {
    "source": "TODO",
    "purpose": "TODO",
    "features": EXPECTED_FEATURES,
    "target": TARGET,
    "split": {"train": 0.60, "validation": 0.20, "test": 0.20, "random_state": RANDOM_STATE},
    "known_limitations": ["TODO", "TODO", "TODO"],
}
risk_register = {
    "version": "s01-lifecycle",
    "risks": [
        {"id": "R1", "risk": "TODO", "impact": "high", "owner": "TODO", "mitigation": "TODO"},
        {"id": "R2", "risk": "TODO", "impact": "high", "owner": "TODO", "mitigation": "TODO"},
        {"id": "R3", "risk": "TODO", "impact": "high", "owner": "TODO", "mitigation": "TODO"},
        {"id": "R4", "risk": "TODO", "impact": "high", "owner": "TODO", "mitigation": "TODO"},
    ],
}
assert len(dataset_card["known_limitations"]) >= 3
assert len(risk_register["risks"]) >= 4

### 6a. Inputs de datos

Crea dos objetos `mlflow.data` con nombres distintos. Comprueba sus nombres antes de lanzar runs.

In [0]:
# Incluye target alineado con cada split y usa DATASET_PATH como source.
train_df = X_train.copy()
train_df[TARGET] = y_train.values
val_df = X_val.copy()
val_df[TARGET] = y_val.values

train_dataset = mlflow.data.from_pandas(
    train_df,
    source=DATASET_PATH,
    name="train_split",
    targets=TARGET,
)
validation_dataset = mlflow.data.from_pandas(
    val_df,
    source=DATASET_PATH,
    name="validation_split",
    targets=TARGET,
)

assert train_dataset.name == "train_split"
assert validation_dataset.name == "validation_split"

/databricks/python/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/databricks/python/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


### 6b. Artefactos del candidato

Implementa una función pequeña que registre firma/input example, `model_info.model_uri`, pickle,
tamaño, matriz de confusión y contrato. Podrás probarla dentro de un único run antes del bucle.

In [0]:
def log_candidate_artifacts(
    model: Pipeline, predictions: np.ndarray, candidate_index: int
) -> str:
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        signature=infer_signature(X_val, model.predict(X_val)),
        input_example=X_val.head(3),
    )
    model_uri = model_info.model_uri
    mlflow.set_tag("candidate.model_uri", model_uri)
    mlflow.set_tag("candidate.index", str(candidate_index))

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)

        # Pickle
        pkl = tmpdir / "model.pkl"
        with open(pkl, "wb") as f:
            pickle.dump(model, f)
        mlflow.log_artifact(str(pkl), artifact_path="deployment")
        mlflow.log_metric("pickle_size_bytes", pkl.stat().st_size)

        # Matriz de confusión
        cm = ConfusionMatrixDisplay.from_predictions(y_val, predictions)
        cm_path = tmpdir / "confusion_matrix.png"
        cm.figure_.savefig(cm_path)
        mlflow.log_artifact(str(cm_path))
        plt.close(cm.figure_)

        # Contrato de inferencia
        contract = {
            "model_name": f"candidate_{candidate_index}",
            "model_uri": model_uri,
            "features": EXPECTED_FEATURES,
            "target": TARGET,
            "input_dtypes": {c: str(X_val[c].dtype) for c in EXPECTED_FEATURES},
            "output": {"prediction": "int64", "prediction_proba": "float64"},
        }
        cpath = tmpdir / "inference_contract.json"
        cpath.write_text(json.dumps(contract, indent=2))
        mlflow.log_artifact(str(cpath))

    return model_uri

### 6c. Runs comparables

Ahora el bucle se ocupa sólo de identidad, parámetros, inputs, gobierno, entrenamiento, métricas
y de llamar al helper de artefactos. Conserva `experiment_id` y cada `run_id`.

In [0]:
candidate_run_ids = []

def _json_default(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(f"Object of type {o.__class__.__name__} is not JSON serializable")

for idx, config in enumerate(CANDIDATES):
    with mlflow.start_run(experiment_id=experiment_id, run_name=config["name"]) as run:
        run_id = run.info.run_id
        candidate_run_ids.append(run_id)

        # Tags de identidad
        mlflow.set_tags({
            "student.alias": STUDENT_ALIAS,
            "batch.id": BATCH_ID,
            "candidate.name": config["name"],
            "candidate.index": str(idx),
            "stage": "candidate",
        })

        # Parámetros
        mlflow.log_params({
            "n_estimators": config.get("n_estimators", 100),
            "max_depth": config.get("max_depth", None),
            "min_samples_leaf": config.get("min_samples_leaf", 1),
            "class_weight": config.get("class_weight", None),
            "random_state": config.get("random_state", RANDOM_STATE),
        })

        # Sets de datos
        mlflow.log_input(train_dataset, context="train")
        mlflow.log_input(validation_dataset, context="validation")

        # dataset_card, quality_report, risk_register
        with tempfile.TemporaryDirectory() as tmpdir:
            tmpdir = Path(tmpdir)
            for fname, payload in [
                ("dataset_card.json", dataset_card),
                ("quality_report.json", quality_report),
                ("risk_register.json", risk_register),
            ]:
                fpath = tmpdir / fname
                fpath.write_text(json.dumps(payload, indent=2, default=_json_default))
                mlflow.log_artifact(str(fpath))

        # Entrenamiento
        model = build_pipeline(config)
        model.fit(X_train, y_train)

        # Métricas de validación
        val_metrics = evaluate_classifier(model, X_val, y_val)
        for metric_name, metric_value in val_metrics.items():
            mlflow.log_metric(f"validation.{metric_name}", metric_value)

        # Classification report
        val_predictions = model.predict(X_val)
        report = classification_report(y_val, val_predictions, zero_division=0)
        with tempfile.TemporaryDirectory() as tmpdir:
            tmpdir = Path(tmpdir)
            report_path = tmpdir / "classification_report.txt"
            report_path.write_text(report)
            mlflow.log_artifact(str(report_path))

        # Artefactos
        log_candidate_artifacts(model, val_predictions, idx)

assert len(candidate_run_ids) == len(CANDIDATES)

2026/09/30 11:08:15 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/09/30 11:08:17 WARNIN

## 7. Busca y compara exclusivamente los runs de tu alias y `BATCH_ID`

In [0]:
# TODO 7: usa mlflow.search_runs con tres tags en filter_string y muestra parámetros,
# métricas de validación, latencia y run_id en un DataFrame `comparison`.
comparison = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string=(
        f'tags."student.alias" = \'{STUDENT_ALIAS}\' '
        f'AND tags."batch.id" = \'{BATCH_ID}\' '
        f'AND tags."stage" = \'candidate\''
    ),
)

# Descartar runs de ejecuciones fallidas
comparison = comparison.sort_values("start_time", ascending=False).drop_duplicates(
    subset=["tags.candidate.name"], keep="first"
)

# run_id, parámetros, métricas de validación
param_cols = [c for c in comparison.columns if c.startswith("params.")]
metric_cols = [c for c in comparison.columns if c.startswith("metrics.validation.")]

comparison = comparison[["run_id"] + sorted(param_cols) + sorted(metric_cols)]
comparison = comparison.sort_values("metrics.validation.f1", ascending=False).reset_index(drop=True)

assert len(comparison) == len(CANDIDATES), f"Se esperaban {len(CANDIDATES)} runs, se hallaron {len(comparison)}"
comparison

,run_id,params.class_weight,params.max_depth,params.min_samples_leaf,params.n_estimators,params.random_state,metrics.validation.accuracy,metrics.validation.f1,metrics.validation.latency_per_row,metrics.validation.precision,metrics.validation.recall,metrics.validation.roc_auc
0,588a9dc9c4194319b4dd45f572be41a8,None,5,1,100,42,0.819672,0.830769,0.000119,0.818182,0.84375,0.877155
1,b53e44530e1848a387c4b379b50b9c09,balanced,15,3,200,42,0.803279,0.818182,0.000163,0.794118,0.84375,0.885776
2,3078ebf30fed4e2abc7d3d842545491e,None,20,1,100,42,0.803279,0.818182,0.000301,0.794118,0.84375,0.884698
3,39271f658619478c84ef3c88018aa2d8,None,None,1,100,42,0.803279,0.818182,0.000119,0.794118,0.84375,0.884698
4,4a5acd401ff3482bae9538c1192466a7,None,10,4,150,42,0.786885,0.805970,0.000135,0.771429,0.84375,0.887931
5,485d71e52c994fd08449ac3e28be596f,None,None,5,100,42,0.786885,0.805970,0.000122,0.771429,0.84375,0.893319
6,79fcbcc8a4c245e084fef3fbced13ed7,balanced,None,1,100,42,0.786885,0.800000,0.000115,0.787879,0.81250,0.871767
7,2e683596d56340809084e275afe02c92,None,None,1,300,42,0.786885,0.800000,0.000218,0.787879,0.81250,0.877694


## 8. Aplica el gate sin mirar test

Filtra por recall mínimo; ordena por F1 DESC, ROC AUC DESC y latencia ASC. Conserva
`BEST_RUN_ID`, `BEST_MODEL_URI` y un `selection_report` que diga explícitamente que test no
participó en la selección.

In [0]:
eligible = comparison[
    comparison["metrics.validation.recall"] >= MIN_VALIDATION_RECALL
].copy()

assert len(eligible) > 0, (
    f"Ningún candidato supera el recall mínimo de validación "
    f"({MIN_VALIDATION_RECALL}). El umbral no se rebaja a posteriori."
)

# Ordenar por F1 DESC, ROC AUC DESC, latencia
eligible = eligible.sort_values(
    by=[
        "metrics.validation.f1",
        "metrics.validation.roc_auc",
        "metrics.validation.latency_per_row",
    ],
    ascending=[False, False, True],
).reset_index(drop=True)

BEST_RUN_ID = eligible.loc[0, "run_id"]
BEST_MODEL_URI = f"runs:/{BEST_RUN_ID}/model"

selection_report = {
    "gate": {
        "metric": "validation.recall",
        "threshold": MIN_VALIDATION_RECALL,
        "candidates_total": len(comparison),
        "candidates_eligible": len(eligible),
    },
    "ranking": {
        "keys": ["validation.f1 DESC", "validation.roc_auc DESC", "validation.latency_per_row ASC"],
    },
    "winner": {
        "run_id": BEST_RUN_ID,
        "model_uri": BEST_MODEL_URI,
        "candidate_name": eligible.loc[0, "tags.candidate.name"]
        if "tags.candidate.name" in eligible.columns
        else None,
        "validation.f1": float(eligible.loc[0, "metrics.validation.f1"]),
        "validation.recall": float(eligible.loc[0, "metrics.validation.recall"]),
        "validation.roc_auc": float(eligible.loc[0, "metrics.validation.roc_auc"]),
        "validation.latency_per_row": float(eligible.loc[0, "metrics.validation.latency_per_row"]),
    },
    "test_participated_in_selection": False,
    "note": ("El conjunto de test NO participó en la selección del ganador. "
             "Se evalúa una sola vez en la sección 9, después de fijar el modelo."),
}

assert BEST_RUN_ID is not None
assert BEST_MODEL_URI.startswith("runs:/")
assert selection_report["test_participated_in_selection"] is False
assert selection_report["gate"]["threshold"] == MIN_VALIDATION_RECALL

print(json.dumps(selection_report, indent=2, default=_json_default))

{
  "gate": {
    "metric": "validation.recall",
    "threshold": 0.72,
    "candidates_total": 8,
    "candidates_eligible": 8
  },
  "ranking": {
    "keys": [
      "validation.f1 DESC",
      "validation.roc_auc DESC",
      "validation.latency_per_row ASC"
    ]
  },
  "winner": {
    "run_id": "588a9dc9c4194319b4dd45f572be41a8",
    "model_uri": "runs:/588a9dc9c4194319b4dd45f572be41a8/model",
    "candidate_name": null,
    "validation.f1": 0.8307692307692308,
    "validation.recall": 0.84375,
    "validation.roc_auc": 0.877155172413793,
    "validation.latency_per_row": 0.00011890590163885895
  },
  "test_participated_in_selection": false,
  "note": "El conjunto de test NO particip\u00f3 en la selecci\u00f3n del ganador. Se eval\u00faa una sola vez en la secci\u00f3n 9, despu\u00e9s de fijar el modelo."
}


## 9. Carga el modelo desde MLflow, evalúa test una sola vez y completa el run ganador

In [0]:
# Cargar modelo ganador
final_model = mlflow.sklearn.load_model(BEST_MODEL_URI)

# Test
test_metrics = evaluate_classifier(final_model, X_test, y_test)

# Abrir el run ganador y registrar lo pendiente
with mlflow.start_run(run_id=BEST_RUN_ID):
    test_df = X_test.copy()
    test_df[TARGET] = y_test.values
    test_dataset = mlflow.data.from_pandas(
        test_df,
        source=DATASET_PATH,
        name="test_split",
        targets=TARGET,
    )
    mlflow.log_input(test_dataset, context="test")

    # Métricas test
    for metric_name, metric_value in test_metrics.items():
        mlflow.log_metric(f"test.{metric_name}", metric_value)

    # Tags
    mlflow.set_tags({
        "selection.gate": "validation.recall >= MIN_VALIDATION_RECALL",
        "selection.ranking": "validation.f1 DESC, validation.roc_auc DESC, validation.latency_per_row ASC",
        "selection.test_participated": "false",
        "selection.winner": "true",
        "stage": "selected",
    })

    # decision.json
    decision = {
        "best_run_id": BEST_RUN_ID,
        "best_model_uri": BEST_MODEL_URI,
        "selection_report": selection_report,
        "test_metrics": test_metrics,
    }
    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)
        decision_path = tmpdir / "decision.json"
        decision_path.write_text(json.dumps(decision, indent=2, default=_json_default))
        mlflow.log_artifact(str(decision_path), artifact_path="selection")

print(json.dumps({"test_metrics": test_metrics, "best_run_id": BEST_RUN_ID}, indent=2, default=_json_default))

/databricks/python/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/d

{
  "test_metrics": {
    "accuracy": 0.8852459016393442,
    "precision": 0.8787878787878788,
    "recall": 0.90625,
    "f1": 0.8923076923076924,
    "roc_auc": 0.9256465517241379,
    "latency_per_row": 0.00014636909835659385
  },
  "best_run_id": "588a9dc9c4194319b4dd45f572be41a8"
}


## 10. Registra el modelo ganador en Unity Catalog

Usa el catálogo y esquema activos, un nombre de tres niveles y tu alias saneado. Añade
descripción y tags. Registra referencia+ganador, asigna `Challenger`, haz smoke test,
promociona, comprueba rollback y deja el ganador como `Champion`. Si recibes `PERMISSION_DENIED`, consulta la
guía: no pegues credenciales ni cambies a recursos de pago.

In [0]:
# TODO 10: set_registry_uri('databricks-uc'), register_model, descripciones,
# tags, Challenger, promoción, rollback, re-promoción y variable champion.
import re
from mlflow import MlflowClient

# Unity Catalog
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient(registry_uri="databricks-uc")

#Nombre de tres niveles
catalog = spark.catalog.currentCatalog()
schema = spark.catalog.currentDatabase()
sanitized_alias = re.sub(r"[^A-Za-z0-9_]", "_", STUDENT_ALIAS.lower())
registered_model_name = f"{catalog}.{schema}.{sanitized_alias}_churn_model"

# Validar artefacto antes de registrar
pyfunc_check = mlflow.pyfunc.load_model(BEST_MODEL_URI)
assert pyfunc_check.metadata.signature is not None, "El modelo no tiene firma."
assert pyfunc_check.input_example is not None, "El modelo no tiene input_example."
mlflow.models.predict(model_uri=BEST_MODEL_URI, input_data=X_val.head(3))
print("Validación del artefacto correcta.")

# Registrar modelo
mv = mlflow.register_model(
    model_uri=BEST_MODEL_URI,
    name=registered_model_name,
    await_registration_for=300,
)
model_version = mv.version
print(f"Registrado: {registered_model_name} v{model_version}")

# Descripciones
client.update_registered_model(
    name=registered_model_name,
    description=(
        f"Clasificador de churn ({STUDENT_ALIAS}, batch {BATCH_ID}). "
        f"Ganador seleccionado con gate de recall de validación ≥ {MIN_VALIDATION_RECALL}. "
        "Pipeline RandomForestClassifier con preprocesamiento embebido en el artefacto."
    ),
)
client.update_model_version(
    name=registered_model_name,
    version=model_version,
    description=f"Registrado desde el run {BEST_RUN_ID}. Ganador del proceso de selección.",
)

# Tags
for key, value in {
    "student_alias": STUDENT_ALIAS,
    "batch_id": BATCH_ID,
    "source_run_id": BEST_RUN_ID,
    "stage": "registered",
}.items():
    client.set_model_version_tag(
        name=registered_model_name, version=model_version, key=key, value=value
    )

# Alias Challenger
client.set_registered_model_alias(
    name=registered_model_name, alias="Challenger", version=model_version
)

# Smoke test con Challenger
challenger = mlflow.pyfunc.load_model(f"models:/{registered_model_name}@Challenger")
_challenger_preds = challenger.predict(X_val.head(3))
assert len(_challenger_preds) == 3
print("Smoke test con Challenger superado.")

# Promocionar a Champion
client.set_registered_model_alias(
    name=registered_model_name, alias="Champion", version=model_version
)

# Verificar rollback
client.delete_registered_model_alias(
    name=registered_model_name, alias="Champion"
)
_load_ok = False
try:
    mlflow.pyfunc.load_model(f"models:/{registered_model_name}@Champion")
    _load_ok = True
except Exception:
    pass
assert not _load_ok, (
    "Rollback fallido: el modelo sigue siendo cargable como @Champion tras eliminar el alias."
)
print("Rollback verificado: @Champion dejó de resolver tras eliminar el alias.")

# Re-promoción a Champion
client.set_registered_model_alias(
    name=registered_model_name, alias="Champion", version=model_version
)

# Smoke test final
champion = mlflow.pyfunc.load_model(f"models:/{registered_model_name}@Champion")
_champion_preds = champion.predict(X_val.head(3))
assert len(_champion_preds) == 3

print(json.dumps({
    "registered_model_name": registered_model_name,
    "model_version": int(model_version),
    "champion_uri": f"models:/{registered_model_name}@Champion",
    "challenger_uri": f"models:/{registered_model_name}@Challenger",
    "source_run_id": BEST_RUN_ID,
    "champion_loaded": champion is not None,
}, indent=2, default=_json_default))

2026/09/30 11:34:31 INFO mlflow.models.python_api: It is highly recommended to use `uv` as the environment manager for predicting with MLflow models as its performance is significantly better than other environment managers. Run `pip install uv` to install uv. See https://docs.astral.sh/uv/getting-started/installation for other installation methods.


2026/09/30 11:34:34 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


2026/09/30 11:34:37 INFO mlflow.utils.virtualenv: Environment /tmp/virtualenv_envs/mlflow-4bfa1453a32f1d960f57094995d698fada99ed43 already exists
2026/09/30 11:34:37 INFO mlflow.utils.environment: === Running command '['bash', '-c', 'source /tmp/virtualenv_envs/mlflow-4bfa1453a32f1d960f57094995d698fada99ed43/bin/activate && python -c ""']'
2026/09/30 11:34:37 INFO mlflow.utils.environment: === Running command '['bash', '-c', 'source /tmp/virtualenv_envs/mlflow-4bfa1453a32f1d960f57094995d698fada99ed43/bin/activate && python /databricks/python/lib/python3.12/site-packages/mlflow/pyfunc/_mlflow_pyfunc_backend_predict.py --model-uri file:///local_disk0/user_tmp_data/spark-e80a7e89-4894-4938-9275-b4/tmprs1wixsl/model --content-type json --input-path /local_disk0/user_tmp_data/spark-e80a7e89-4894-4938-9275-b4/tmp8b5lbmgi/input.json']'


{"predictions": [0, 1, 1]}Validación del artefacto correcta.


Registered model 'workspace.default.carrascoalvarezgonzalo_gmail_com_churn_model' already exists. Creating a new version of this model...
2026/09/30 11:34:44 WARNING mlflow.tracking._model_registry.fluent: Run with id 588a9dc9c4194319b4dd45f572be41a8 has no artifacts at artifact path 'model', registering model based on models:/m-a2eadf84fae34db7afcad55d5cd8b431 instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '3' of model 'workspace.default.carrascoalvarezgonzalo_gmail_com_churn_model': https://dbc-57ff49ab-cafb.cloud.databricks.com/explore/data/models/workspace/default/carrascoalvarezgonzalo_gmail_com_churn_model/version/3?o=7474649088606594


Registrado: workspace.default.carrascoalvarezgonzalo_gmail_com_churn_model v3


Smoke test con Challenger superado.
Rollback verificado: @Champion dejó de resolver tras eliminar el alias.


{
  "registered_model_name": "workspace.default.carrascoalvarezgonzalo_gmail_com_churn_model",
  "model_version": 3,
  "champion_uri": "models:/workspace.default.carrascoalvarezgonzalo_gmail_com_churn_model@Champion",
  "challenger_uri": "models:/workspace.default.carrascoalvarezgonzalo_gmail_com_churn_model@Challenger",
  "source_run_id": "588a9dc9c4194319b4dd45f572be41a8",
  "champion_loaded": true
}


## 11. Carga `models:/<nombre>@Champion` y realiza un smoke test con tres filas

In [0]:
champion_model = mlflow.pyfunc.load_model(f"models:/{registered_model_name}@Champion")
smoke_rows = X_val.head(3)
smoke_predictions = champion_model.predict(smoke_rows)

assert len(smoke_predictions) == 3, f"Se esperaban 3 predicciones, se obtuvieron {len(smoke_predictions)}"

print(json.dumps({
    "registered_model_name": registered_model_name,
    "alias": "Champion",
    "model_version": int(model_version),
    "n_predictions": len(smoke_predictions),
    "predictions": smoke_predictions.tolist(),
}, indent=2, default=_json_default))

{
  "registered_model_name": "workspace.default.carrascoalvarezgonzalo_gmail_com_churn_model",
  "alias": "Champion",
  "model_version": 3,
  "n_predictions": 3,
  "predictions": [
    0,
    1,
    1
  ]
}


## 12. API local con el pickle ganador

Resuelve el run de `Champion`, descarga `deployment/model.pkl` y levanta un
`ThreadingHTTPServer` en `127.0.0.1` con puerto automático. Implementa `GET /health` y
`POST /predict`; valida `instances`, columnas ausentes y columnas extra.

In [0]:
# run_id del Champion
champion_version_info = client.get_model_version_by_alias(
    name=registered_model_name, alias="Champion"
)
champion_run_id = champion_version_info.run_id
assert champion_run_id is not None, "No se pudo resolver el run_id del Champion."

# Descargar deployment/model.pkl
with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir = Path(tmpdir)
    client.download_artifacts(
        run_id=champion_run_id, path="deployment/model.pkl", dst_path=str(tmpdir)
    )
    pkl_path = tmpdir / "model.pkl"
    assert pkl_path.exists(), f"No se encontró deployment/model.pkl en el run {champion_run_id}"

    with open(pkl_path, "rb") as f:
        api_model = pickle.load(f)
    _pickle_bytes = pkl_path.stat().st_size

# Comprobar que predice una fila
_sample_row = X_val.head(1)
_pred = api_model.predict(_sample_row)
assert len(_pred) == 1, f"Se esperaba 1 predicción, se obtuvieron {len(_pred)}"

print(json.dumps({
    "champion_run_id": champion_run_id,
    "pickle_bytes": _pickle_bytes,
    "prediction_sample": _pred.tolist(),
}, indent=2, default=_json_default))

{
  "champion_run_id": "588a9dc9c4194319b4dd45f572be41a8",
  "pickle_bytes": 345132,
  "prediction_sample": [
    0
  ]
}


### 12b. Handler y contrato

Define `PredictionHandler` con `_send_json`, `GET /health` y `POST /predict`. Valida objeto raíz,
lista no vacía de objetos, columnas exactas y errores 400. Instrumenta contadores y latencias.

In [0]:
API_STATS = {"requests_total": 0, "requests_success": 0, "requests_rejected": 0, "requests_error": 0, "latencies_ms": []}
API_STATS_LOCK = threading.Lock()

class PredictionHandler(BaseHTTPRequestHandler):
    """Handler HTTP para /health y /predict con instrumentación de API_STATS."""

    def log_message(self, fmt, *args):  # silenciar logs por defecto
        pass

    def _send_json(self, status_code: int, payload: dict):
        body = json.dumps(payload).encode("utf-8")
        self.send_response(status_code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def _record_stat(self, key: str, latency_ms: float | None = None):
        with API_STATS_LOCK:
            API_STATS[key] += 1
            if latency_ms is not None:
                API_STATS["latencies_ms"].append(latency_ms)

    def do_GET(self):
        start = time.perf_counter()
        if self.path == "/health":
            with API_STATS_LOCK:
                snapshot = {
                    "status": "ok",
                    "requests_total": API_STATS["requests_total"],
                    "requests_success": API_STATS["requests_success"],
                    "requests_rejected": API_STATS["requests_rejected"],
                    "requests_error": API_STATS["requests_error"],
                }
            self._send_json(200, snapshot)
        else:
            self._send_json(404, {"error": "not_found", "path": self.path})
        self._record_stat("requests_total", (time.perf_counter() - start) * 1000)

    def do_POST(self):
        start = time.perf_counter()

        def _finish(code: int, payload: dict, stat_key: str):
            latency_ms = (time.perf_counter() - start) * 1000
            self._send_json(code, payload)
            self._record_stat(stat_key, latency_ms)
            self._record_stat("requests_total")

        if self.path != "/predict":
            _finish(404, {"error": "not_found", "path": self.path}, "requests_error")
            return

        # Leer cuerpo
        try:
            length = int(self.headers.get("Content-Length", 0))
            raw = self.rfile.read(length) if length else b""
            payload = json.loads(raw)
        except (ValueError, json.JSONDecodeError):
            _finish(400, {"error": "invalid_json", "detail": "El cuerpo no es JSON válido."},
                    "requests_rejected")
            return

        # Validar objeto payload
        if not isinstance(payload, dict):
            _finish(400, {"error": "invalid_root", "detail": "La raíz debe ser un objeto JSON."},
                    "requests_rejected")
            return

        # Validar instances
        instances = payload.get("instances")
        if not isinstance(instances, list) or len(instances) == 0:
            _finish(400, {"error": "invalid_instances",
                          "detail": "'instances' debe ser una lista no vacía de objetos."},
                    "requests_rejected")
            return

        # Validar que cada instancia es un dict con las columnas exactas
        for i, row in enumerate(instances):
            if not isinstance(row, dict):
                _finish(400, {"error": "invalid_row",
                              "detail": f"instances[{i}] no es un objeto."},
                        "requests_rejected")
                return
            missing = set(EXPECTED_FEATURES) - set(row.keys())
            extra = set(row.keys()) - set(EXPECTED_FEATURES)
            if missing or extra:
                detail_parts = []
                if missing:
                    detail_parts.append(f"columnas_faltantes: {sorted(missing)}")
                if extra:
                    detail_parts.append(f"columnas_extra: {sorted(extra)}")
                _finish(400, {"error": "schema_mismatch",
                              "detail": f"instances[{i}] -> {'; '.join(detail_parts)}"},
                        "requests_rejected")
                return
        # Construir DataFrame y predecir
        try:
            input_df = pd.DataFrame(instances, columns=EXPECTED_FEATURES)
            predictions = api_model.predict(input_df)
            _finish(200, {"predictions": predictions.tolist()}, "requests_success")
        except Exception as exc:
            _finish(500, {"error": "prediction_failed", "detail": str(exc)},
                    "requests_error")

### 12c. Arranque y limpieza idempotente

Arranca en `127.0.0.1` y puerto 0. Define `stop_api()` antes de probar para que un `finally` pueda
cerrar servidor, socket e hilo. No uses `0.0.0.0`.

In [0]:

API_STOPPED = False
api_server = None
API_PORT = None
api_thread = None

def start_api():
    global api_server, API_PORT, api_thread
    api_server = ThreadingHTTPServer(("127.0.0.1", 0), PredictionHandler)
    API_PORT = api_server.server_address[1]
    api_thread = threading.Thread(
        target=api_server.serve_forever,
        daemon=True,
        name="api-server-thread",
    )
    api_thread.start()
    return api_server, API_PORT, api_thread

def stop_api():
    """Cierra servidor, socket e hilo."""
    global API_STOPPED, api_server, api_thread
    if API_STOPPED:
        return
    API_STOPPED = True
    if api_server is not None:
        try:
            api_server.shutdown()
        except Exception:
            pass
        try:
            api_server.server_close()
        except Exception:
            pass
    if api_thread is not None and api_thread is not threading.current_thread():
        api_thread.join(timeout=5)
    print(json.dumps({"stopped": True, "port": API_PORT}, indent=2))

# Arrancar ahora para que las celdas siguientes puedan usar la API
start_api()

assert api_server is not None, "El servidor no se arrancó."
assert API_PORT is not None, "No se asignó un puerto."
assert api_thread is not None and api_thread.is_alive(), "El hilo del servidor no está activo."
assert api_server.server_address[0] == "127.0.0.1", "El servidor no escucha en 127.0.0.1."

print(json.dumps({
    "host": "127.0.0.1",
    "port": API_PORT,
    "thread_alive": api_thread.is_alive(),
}, indent=2))

{
  "host": "127.0.0.1",
  "port": 40805,
  "thread_alive": true
}


## 13. Pruebas de contrato y de error

In [0]:
import urllib.request
import urllib.error


def call_json(method: str, path: str, body=None):
    """Lanza una petición HTTP a la API local y devuelve (status_code, json_response)."""
    url = f"http://127.0.0.1:{API_PORT}{path}"
    data = None
    if body is not None:
        data = json.dumps(body).encode("utf-8")
    req = urllib.request.Request(
        url, data=data, headers={"Content-Type": "application/json"}, method=method
    )
    try:
        with urllib.request.urlopen(req) as resp:
            return resp.status, json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        return exc.code, json.loads(exc.read().decode("utf-8"))


if API_STOPPED:
    API_STOPPED = False
    start_api()

try:
    # GET /health
    status, body = call_json("GET", "/health")
    assert status == 200, f"GET /health -> {status}, se esperaba 200"
    assert body["status"] == "ok", f"GET /health body inesperado: {body}"
    print(f"GET /health -> {status}: {body}")

    # POST válido
    valid_instances = X_val.head(2)[EXPECTED_FEATURES].to_dict(orient="records")
    status, body = call_json("POST", "/predict", {"instances": valid_instances})
    assert status == 200, f"POST válido -> {status}, se esperaba 200"
    assert "predictions" in body, f"POST válido body inesperado: {body}"
    assert len(body["predictions"]) == 2, f"Se esperaban 2 predicciones, se obtuvieron {len(body['predictions'])}"
    print(f"POST válido -> {status}: {body}")

    # Sólo age -> 400 (columnas faltantes)
    status, body = call_json("POST", "/predict", {"instances": [{"age": 30}]})
    assert status == 400, f"POST sólo age -> {status}, se esperaba 400"
    assert body["error"] == "schema_mismatch", f"POST sólo age error inesperado: {body}"
    print(f"POST sólo age -> {status}: {body}")

    # JSON no-object -> 400
    status, body = call_json("POST", "/predict", [1, 2, 3])
    assert status == 400, f"POST JSON no-object -> {status}, se esperaba 400"
    assert body["error"] == "invalid_root", f"POST JSON no-object error inesperado: {body}"
    print(f"POST JSON no-object -> {status}: {body}")

    print(json.dumps({"all_tests_passed": True}, indent=2))
finally:
    stop_api()

GET /health -> 200: {'status': 'ok', 'requests_total': 8, 'requests_success': 2, 'requests_rejected': 4, 'requests_error': 0}
POST válido -> 200: {'predictions': [0, 1]}
POST sólo age -> 400: {'error': 'schema_mismatch', 'detail': "instances[0] -> columnas_faltantes: ['ca', 'chol', 'cp', 'exang', 'fbs', 'oldpeak', 'restecg', 'sex', 'slope', 'thal', 'thalach', 'trestbps']"}
POST JSON no-object -> 400: {'error': 'invalid_root', 'detail': 'La raíz debe ser un objeto JSON.'}
{
  "all_tests_passed": true
}
{
  "stopped": true,
  "port": 33545
}


## 14. Registra un run separado para la evidencia del despliegue

In [0]:
# Run para la evidencia del despliegue
with mlflow.start_run(
    experiment_id=experiment_id,
    run_name=f"deploy-{STUDENT_ALIAS}-{BATCH_ID}",
) as deploy_run:
    DEPLOYMENT_RUN_ID = deploy_run.info.run_id

    # Tags de origen
    mlflow.set_tags({
        "student.alias": STUDENT_ALIAS,
        "batch.id": BATCH_ID,
        "stage": "deployment",
        "deployment.source_run_id": champion_run_id,
        "deployment.registered_model": registered_model_name,
        "deployment.model_version": str(model_version),
        "deployment.alias": "Champion",
    })

    # Métricas de observabilidad
    mlflow.log_metric("deployment.requests_total", API_STATS["requests_total"])
    mlflow.log_metric("deployment.requests_success", API_STATS["requests_success"])
    mlflow.log_metric("deployment.requests_rejected", API_STATS["requests_rejected"])
    mlflow.log_metric("deployment.requests_error", API_STATS["requests_error"])

    _latencies = API_STATS["latencies_ms"]
    if _latencies:
        mlflow.log_metric("deployment.latency_p50_ms", float(np.percentile(_latencies, 50)))
        mlflow.log_metric("deployment.latency_p95_ms", float(np.percentile(_latencies, 95)))
        mlflow.log_metric("deployment.latency_mean_ms", float(np.mean(_latencies)))
        mlflow.log_metric("deployment.latency_max_ms", float(np.max(_latencies)))
    else:
        mlflow.log_metric("deployment.latency_p50_ms", 0.0)
        mlflow.log_metric("deployment.latency_p95_ms", 0.0)
        mlflow.log_metric("deployment.latency_mean_ms", 0.0)
        mlflow.log_metric("deployment.latency_max_ms", 0.0)

    # evidence.json
    evidence = {
        "deployment_run_id": DEPLOYMENT_RUN_ID,
        "source_run_id": champion_run_id,
        "registered_model_name": registered_model_name,
        "model_version": int(model_version),
        "alias": "Champion",
        "student_alias": STUDENT_ALIAS,
        "batch_id": BATCH_ID,
        "api_stats": {
            "requests_total": API_STATS["requests_total"],
            "requests_success": API_STATS["requests_success"],
            "requests_rejected": API_STATS["requests_rejected"],
            "requests_error": API_STATS["requests_error"],
            "latencies_ms": _latencies,
        },
        "note": ("No se registra el payload de entrada para evitar exponer "
                 "información sensible."),
    }

    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir = Path(tmpdir)
        evidence_path = tmpdir / "evidence.json"
        evidence_path.write_text(
            json.dumps(evidence, indent=2, default=_json_default)
        )
        mlflow.log_artifact(str(evidence_path), artifact_path="deployment")

assert DEPLOYMENT_RUN_ID is not None

print(json.dumps({
    "deployment_run_id": DEPLOYMENT_RUN_ID,
    "source_run_id": champion_run_id,
    "registered_model_name": registered_model_name,
    "model_version": int(model_version),
    "alias": "Champion",
    "requests_total": API_STATS["requests_total"],
    "requests_success": API_STATS["requests_success"],
    "requests_rejected": API_STATS["requests_rejected"],
    "requests_error": API_STATS["requests_error"],
}, indent=2, default=_json_default))

2026/09/30 12:00:23 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


{
  "deployment_run_id": "bd6963e80907472ba343d80a70e6371f",
  "source_run_id": "588a9dc9c4194319b4dd45f572be41a8",
  "registered_model_name": "workspace.default.carrascoalvarezgonzalo_gmail_com_churn_model",
  "model_version": 3,
  "alias": "Champion",
  "requests_total": 12,
  "requests_success": 3,
  "requests_rejected": 6,
  "requests_error": 0
}


## 15. Apaga la API y entrega el resumen del ciclo

In [0]:

stop_api()

# Asegurarse de que el servidor ya no está activo
if api_thread is not None:
    assert not api_thread.is_alive(), "El hilo del servidor sigue activo tras el cierre."

# Resumen del ciclo
print(json.dumps({
    "experiment_id": experiment_id,
    "batch_id": BATCH_ID,
    "student_alias": STUDENT_ALIAS,
    "winner": {
        "best_run_id": BEST_RUN_ID,
        "best_model_uri": BEST_MODEL_URI,
        "registered_model_name": registered_model_name,
        "model_version": int(model_version),
        "alias": "Champion",
    },
    "deployment_run_id": DEPLOYMENT_RUN_ID,
}, indent=2, default=_json_default))

{
  "experiment_id": "3733454106584879",
  "batch_id": "acf33af9",
  "student_alias": "carrascoalvarezgonzalo@gmail.com",
  "winner": {
    "best_run_id": "588a9dc9c4194319b4dd45f572be41a8",
    "best_model_uri": "runs:/588a9dc9c4194319b4dd45f572be41a8/model",
    "registered_model_name": "workspace.default.carrascoalvarezgonzalo_gmail_com_churn_model",
    "model_version": 3,
    "alias": "Champion"
  },
  "deployment_run_id": "bd6963e80907472ba343d80a70e6371f"
}
